In [1]:
import psycopg2
import datetime as dt
import numpy as np
import pandas as pd
import logging
import json

In [2]:
# Configure logging
logging.basicConfig(filename='model_training.log', level=logging.DEBUG, 
                    format='%(asctime)s %(levelname)s:%(message)s')

In [3]:
# Load database configuration from JSON
with open('../../db_config.json') as config_file:
    db_params = json.load(config_file)
 
# Connect to the database
try:
    conn = psycopg2.connect(**db_params)
    cursor = conn.cursor()
    logging.info('Connection to database established')
except OperationalError as e:
    logging.error('Connection to database could not be established.')

INFO: Connection to database established


## Second table

In [4]:
create_table_2 = """
DROP TABLE IF EXISTS group14_warehouse.regression_data;
CREATE TABLE IF NOT EXISTS group14_warehouse.regression_data AS
SELECT 
    safe.event_start,
    safe.event_end,
    RTRIM(safe.category) AS event_cat, 
    RTRIM(safe.incident_severity) AS event_sev, 
    safe.speed_kmh,
    safe.end_speed_kmh,
    safe.maxwaarde, 
    safe.road_name AS streetname, 
    prec.RI_PWS_10 AS rain_intensity,
    temp.T_DRYB_10 AS temperature,
    wind.FF_SENSOR_10 AS windspeed,
    SUBSTRING(safe.speed_limit, 1, 2) AS speed_limit,
    safe.light_condition,
    safe.accident_severity,
    safe.accident_probability
FROM 
    group14_warehouse.accident_incident_merged_table safe
JOIN 
-- Rounding the safe driving timestamp to nearest 10 minute to join with precipitation datset
    data_lake.precipitation prec ON TO_TIMESTAMP(ROUND(EXTRACT(EPOCH FROM safe.event_start) / 600) * 600) = prec.dtg
JOIN 
    data_lake.temperature temp ON prec.dtg = temp.dtg
JOIN 
    data_lake.wind wind ON prec.dtg = wind.dtg
WHERE 
    prec.RI_PWS_10 IS NOT NULL
    AND temp.T_DRYB_10 IS NOT NULL
    AND wind.FF_SENSOR_10 IS NOT NULL
    AND safe.is_valid = TRUE
    AND prec.RI_PWS_10 - prec.RI_REGENM_10 < 1;
"""    

In [5]:
cursor.execute(create_table_2)

In [6]:
conn.commit()
logging.info('Table 2 created and pushed to data warehouse.')

INFO: Table 2 created and pushed to data warehouse.


In [7]:
cursor.execute('SELECT * FROM group14_warehouse.regression_data')
table_1 = pd.DataFrame(cursor.fetchall(), columns=['datetime_s', 'datetime_e', 'event_cat', 'event_sev', 'speed', 'end_speed',
                                                   'maxwaarde', 'streetname', 'rain_intensity', 'temperature', 'windspeed',
                                                   'speed_limit', 'light_condition', 'accident_sev', 'accident_prob'])
table_1['datetime_s'] = pd.to_datetime(table_1['datetime_s'])
table_1['datetime_e'] = pd.to_datetime(table_1['datetime_e'])

table_1.head(10)

,datetime_s,datetime_e,event_cat,event_sev,speed,end_speed,maxwaarde,streetname,rain_intensity,temperature,windspeed,speed_limit,light_condition,accident_sev,accident_prob
0,2018-01-01 00:21:41.100,2018-01-01 00:21:43.500,HARSH CORNERING,HC1,30.577536,37.014910,0.854562,Doornboslaan,0.0,8.8,12.87,Un,Daylight,Material Damage Only,3.467174
1,2018-01-01 00:18:20.500,2018-01-01 00:18:28.500,SPEED,SP1,82.076546,82.076546,86.904580,Backer en Ruebweg,0.0,8.8,12.87,Un,Darkness,Material Damage Only,3.049560
2,2018-01-01 00:18:20.500,2018-01-01 00:18:28.500,SPEED,SP1,82.076546,82.076546,86.904580,Backer en Ruebweg,0.0,8.8,12.87,Un,Daylight,Material Damage Only,3.049560
3,2018-01-01 00:18:20.500,2018-01-01 00:18:28.500,SPEED,SP1,82.076546,82.076546,86.904580,Backer en Ruebweg,0.0,8.8,12.87,Un,Daylight,Material Damage Only,3.049560
4,2018-01-01 00:18:20.500,2018-01-01 00:18:28.500,SPEED,SP1,82.076546,82.076546,86.904580,Backer en Ruebweg,0.0,8.8,12.87,70,Daylight,Material Damage Only,3.049560
5,2018-01-01 00:18:20.500,2018-01-01 00:18:28.500,SPEED,SP1,82.076546,82.076546,86.904580,Backer en Ruebweg,0.0,8.8,12.87,70,Daylight,Material Damage Only,3.049560
6,2018-01-01 00:21:41.100,2018-01-01 00:21:43.500,HARSH CORNERING,HC1,30.577536,37.014910,0.854562,Doornboslaan,0.0,8.8,12.87,Un,Daylight,Material Damage Only,3.467174
7,2018-01-01 00:21:41.100,2018-01-01 00:21:43.500,HARSH CORNERING,HC1,30.577536,37.014910,0.854562,Doornboslaan,0.0,8.8,12.87,70,Daylight,Material Damage Only,3.467174
8,2018-01-01 00:21:41.100,2018-01-01 00:21:43.500,HARSH CORNERING,HC1,30.577536,37.014910,0.854562,Doornboslaan,0.0,8.8,12.87,50,Twilight,Material Damage Only,3.467174
9,2018-01-01 00:21:41.100,2018-01-01 00:21:43.500,HARSH CORNERING,HC1,30.577536,37.014910,0.854562,Doornboslaan,0.0,8.8,12.87,Un,Darkness,Material Damage Only,3.467174


In [8]:
conn.close()
logging.info('Connection to database closed.')

INFO: Connection to database closed.


In [9]:
len(table_1)

5929588